# xG-Spark: Expected Goals with Apache Spark
## Phase 2: Ingestion (Bronze Layer)

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent          # this notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from src.config import BRONZE_DIR
from src.spark_session import get_spark

spark = get_spark("xG-notebook")
spark


In [4]:
competitions = spark.read.parquet((BRONZE_DIR / "competitions").as_posix())
matches = spark.read.parquet((BRONZE_DIR / "matches").as_posix())
lineups = spark.read.parquet((BRONZE_DIR / "lineups").as_posix())
events = spark.read.parquet((BRONZE_DIR / "events").as_posix())

for name, df in [("competitions", competitions), ("matches", matches),
                 ("lineups", lineups), ("events", events)]:
    print(f"{name:<13} {df.count():>10,} rows  {len(df.columns):>3} columns")


competitions          80 rows   12 columns
matches            1,781 rows   18 columns
lineups            3,562 rows    4 columns
events         6,279,749 rows   45 columns


In [5]:
(events.groupBy("competition_id", "season_id").count()
    .join(competitions.select("competition_id", "season_id",
                              "competition_name", "season_name"),
          on=["competition_id", "season_id"])
    .orderBy(F.desc("count"))
    .show(truncate=False))


+--------------+---------+-------+----------------+-----------+
|competition_id|season_id|count  |competition_name|season_name|
+--------------+---------+-------+----------------+-----------+
|7             |27       |1358593|Ligue 1         |2015/2016  |
|12            |27       |1353739|Serie A         |2015/2016  |
|2             |27       |1313773|Premier League  |2015/2016  |
|11            |27       |1295354|La Liga         |2015/2016  |
|43            |106      |234637 |FIFA World Cup  |2022       |
|43            |3        |227825 |FIFA World Cup  |2018       |
|55            |43       |192664 |UEFA Euro       |2020       |
|55            |282      |187924 |UEFA Euro       |2024       |
|9             |27       |115240 |1. Bundesliga   |2015/2016  |
+--------------+---------+-------+----------------+-----------+



In [6]:
(events.groupBy(F.col("type.name").alias("event_type"))
    .count()
    .orderBy(F.desc("count"))
    .show(15))


+--------------+-------+
|    event_type|  count|
+--------------+-------+
|          Pass|1752850|
| Ball Receipt*|1641275|
|         Carry|1331430|
|      Pressure| 573997|
| Ball Recovery| 186672|
|          Duel| 142274|
|     Clearance|  83524|
|         Block|  69045|
|       Dribble|  60195|
|   Goal Keeper|  54308|
|Foul Committed|  53022|
|    Miscontrol|  51445|
|      Foul Won|  50446|
|  Dispossessed|  45599|
|          Shot|  44557|
+--------------+-------+
only showing top 15 rows



In [7]:
events.select("shot").printSchema()


root
 |-- shot: struct (nullable = true)
 |    |-- aerial_won: boolean (nullable = true)
 |    |-- body_part: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- deflected: boolean (nullable = true)
 |    |-- end_location: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- first_time: boolean (nullable = true)
 |    |-- follows_dribble: boolean (nullable = true)
 |    |-- freeze_frame: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- location: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- player: struct (nullable = true)
 |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- position: struct (nullable = true)
 |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- name: string (nul

In [8]:
(events.filter(F.col("type.name") == "Shot")
    .select("match_id", "minute",
            F.col("player.name").alias("player"),
            "location",
            F.col("shot.outcome.name").alias("outcome"),
            F.col("shot.statsbomb_xg").alias("statsbomb_xg"))
    .show(10, truncate=False))


+--------+------+--------------------------+-------------+-------+------------+
|match_id|minute|player                    |location     |outcome|statsbomb_xg|
+--------+------+--------------------------+-------------+-------+------------+
|3795108 |1     |Xherdan Shaqiri           |[91.9, 39.0] |Blocked|0.030168695 |
|3795108 |7     |Jordi Alba Ramos          |[96.5, 29.7] |Blocked|0.021315211 |
|3795108 |16    |Jorge Resurrección Merodio|[97.8, 25.9] |Off T  |0.047888376 |
|3795108 |24    |César Azpilicueta Tanco   |[112.0, 40.7]|Saved  |0.10035629  |
|3795108 |33    |Haris Seferović           |[99.7, 58.6] |Blocked|0.008757749 |
|3795108 |33    |Manuel Obafemi Akanji     |[113.2, 39.9]|Off T  |0.08995936  |
|3795108 |38    |Silvan Widmer             |[112.9, 34.3]|Off T  |0.06444093  |
|3795108 |46    |Daniel Olmo Carvajal      |[104.9, 30.2]|Saved  |0.08699033  |
|3795108 |47    |Manuel Obafemi Akanji     |[111.9, 38.8]|Wayward|0.11480155  |
|3795108 |49    |Jorge Resurrección Mero

In [9]:
print("Partitions in events DataFrame:", events.rdd.getNumPartitions())

# Look for "PartitionFilters" in the plan: Spark reads ONLY the Premier League folder
events.filter(F.col("competition_id") == 2).select("id").explain()


Partitions in events DataFrame: 23
== Physical Plan ==
*(1) Project [id#85]
+- *(1) ColumnarToRow
   +- FileScan parquet [id#85,competition_id#111,season_id#112] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/adity/Desktop/xg_spark/data/bronze/events], PartitionFilters: [isnotnull(competition_id#111), (competition_id#111 = 2)], PushedFilters: [], ReadSchema: struct<id:string>


